<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/02-visualizacao_resultados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 1: Configuração e Criação da Pasta de Exportação
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from google.colab import drive
import os

# 1. Montagem do Drive
drive.mount('/content/drive')

# 2. Definição de Caminhos (AJUSTE O CAMINHO DO SEU BANCO ABAIXO)
DB_PATH = '/content/drive/My Drive/mba-engsof-tcc/v5/base-dados-v5.db'
EXPORT_PATH = '/content/drive/My Drive/mba-engsof-tcc/v5/graficos-tcc'

# Cria a pasta de exportação se ela não existir
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📂 Pasta criada: {EXPORT_PATH}")

def get_connection():
    return sqlite3.connect(DB_PATH)

# Garante suporte a acentuação e visual limpo
sns.set_context("paper", font_scale=1.2)

# Configurações para qualidade ds imagens
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("✅ Ambiente configurado para exportação de imagens JPG.")

In [ ]:
# Célula 2: Geração de Nuvens de Palavras (Apenas Versos Positivos com Diferenciação por Autor)
import nltk
from nltk.corpus import stopwords
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import os

nltk.download('stopwords')

def executar_nuvens_antidotos_positivos_refinada():
    conn = get_connection()

    # 1. Seleciona os eixos (excluindo o narrativo)
    query_antidotos = "SELECT id, antidoto_referencia FROM topico WHERE id != 3"
    df_topicos = pd.read_sql_query(query_antidotos, conn)

    # 2. Stopwords: Limpeza de resíduos geográficos e estruturais bíblicos
    stop_words_pt = set(stopwords.words('portuguese'))
    custom_stops = {
        'disse', 'então', 'veio', 'porque', 'pois', 'sobre', 'todos', 'tudo',
        'assim', 'ainda', 'outra', 'outros', 'será', 'pode', 'fazer', 'tão',
        'casa', 'filho', 'filhos', 'homem', 'mulher', 'terra', 'povo', 'rei',
        'senhor', 'deus', 'jesus', 'cristo', 'amém', 'ora', 'eis', 'vós',
        'teu', 'tua', 'meu', 'minha', 'toda', 'ano', 'anos', 'morreu', 'mortos',
        # Adicionando ruídos geográficos/históricos comuns que "sujam" a análise existencial
        'israel', 'judá', 'jerusalém', 'egito', 'babilônia', 'filisteus', 'moisés', 'davi'
    }
    todas_stops = stop_words_pt.union(custom_stops)

    # 3. Dicionário de Cores para Identidade Visual do TCC
    # Han = Tons de Verde/Ciano (Descanso/Vigor)
    # Bauman = Tons de Outono/Laranja (Solidez/Rocha)
    # Frankl = Tons de Azul/Roxo (Profundidade/Sentido)
    cores_eixos = {
        0: 'viridis',      # Esgotamento (Han)
        1: 'YlOrBr',       # Transitoriedade (Bauman)
        2: 'coolwarm'      # Insignificância (Frankl)
    }

    print("🌟 Gerando Nuvens de Antídotos (Filtro: Sentimento Positivo)...")

    for _, row in df_topicos.iterrows():
        t_id = row['id']
        nome_antidoto = row['antidoto_referencia']

        # Consulta com filtro de sentimento POSITIVO
        query_texto = f"""
            SELECT vl.texto_limpo
            FROM verso_limpo vl
            JOIN verso_topico vt ON vl.verso_id = vt.verso_id
            JOIN verso_sentimento s ON vl.verso_id = s.verso_id
            WHERE vt.topico_id = {t_id}
            AND s.sentimento_num = 1
        """
        df_textos = pd.read_sql_query(query_texto, conn)

        if not df_textos.empty:
            texto_final = " ".join(df_textos['texto_limpo'].fillna('').tolist())

            # Configuração estética focada em legibilidade acadêmica
            wordcloud = WordCloud(width=1600, height=900,
                                  background_color='white',
                                  max_words=60, # Reduzido para destacar apenas o essencial
                                  stopwords=todas_stops,
                                  colormap=cores_eixos.get(t_id, 'plasma'),
                                  collocations=False,
                                  prefer_horizontal=0.85).generate(texto_final)

            plt.figure(figsize=(14, 8))
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis('off')

            # Título interno opcional (comentado caso queira o gráfico limpo para o Word)
            # plt.title(f"Antídotos: {nome_antidoto}", fontsize=18, pad=20)

            # Nome de arquivo limpo e padronizado
            safe_name = nome_antidoto.split('(')[0].strip().lower().replace(' ', '_')
            file_name = f"nuvem_cura_{safe_name}.jpg"

            plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
            plt.close()
            print(f"✅ Nuvem de 'Cura' salva: {file_name} (Esquema de cores: {cores_eixos.get(t_id)})")
        else:
            print(f"⚠️ Versos positivos insuficientes para: {nome_antidoto}")

    conn.close()

executar_nuvens_antidotos_positivos_refinada()

In [ ]:
# Célula 3: Distribuição de ANTÍDOTOS (Positivos) por Gênero Literário
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_grafico_antidotos_por_genero_filtrado():
    conn = get_connection()

    # 1. Consulta SQL Refinada: Apenas Versos POSITIVOS (Antídotos)
    query = """
        SELECT
            g.nome as Genero,
            t.antidoto_referencia as Antidoto,
            COUNT(vt.verso_id) as Frequencia
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso v ON vt.verso_id = v.id
        JOIN livro l ON v.livro_id = l.id
        JOIN genero_literario g ON l.genero_id = g.id
        WHERE vs.sentimento_num = 1  -- FILTRO CRUCIAL: Apenas a Cura
          AND t.id != 3              -- Exclui Narrativo/Outros
        GROUP BY Genero, Antidoto
    """

    df_dist = pd.read_sql_query(query, conn)
    conn.close()

    if df_dist.empty:
        print("⚠️ Dados não encontrados. Verifique se a Célula 6 (Sentimentos) foi executada.")
        return

    # 2. Pivotar e Ordenar
    df_pivot = df_dist.pivot(index='Genero', columns='Antidoto', values='Frequencia').fillna(0)

    # Ordenar pelo total de antídotos para um visual mais organizado (escadinha)
    df_pivot['Total'] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values(by='Total', ascending=True).drop(columns='Total')

    # 3. Configuração Estética
    sns.set_style("white")

    # Paleta de cores harmonizada com as Nuvens de Palavras
    # Azul (Frankl), Verde (Han), Laranja (Bauman)
    cores_harmonizadas = ['#2E8B57', '#4682B4', '#D2691E']

    ax = df_pivot.plot(kind='barh',
                       stacked=True,
                       color=cores_harmonizadas,
                       figsize=(14, 8),
                       width=0.75,
                       edgecolor='white',
                       linewidth=1)

    # Rótulos
    plt.xlabel('Volume de Antídotos (Versículos Positivos)', fontsize=12, labelpad=15)
    plt.ylabel('Gênero Literário', fontsize=12, labelpad=15)

    # Adicionando os valores numéricos dentro ou fora das barras para facilitar a leitura da banca
    for p in ax.patches:
        width = p.get_width()
        if width > 5: # Só desenha se a barra for visível
            ax.annotate(f'{int(width)}',
                        (p.get_x() + width / 2, p.get_y() + p.get_height() / 2),
                        ha='center', va='center',
                        fontsize=9, color='white', fontweight='bold')

    sns.despine(left=False, bottom=False)
    plt.legend(title='Eixos Existenciais', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    plt.tight_layout()

    # 4. Exportação
    file_name = "distribuicao_antidotos_genero_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')

    print(f"✅ Gráfico de Antídotos salvo com sucesso: {file_name}")
    plt.show()

gerar_grafico_antidotos_por_genero_filtrado()

In [ ]:
# Célula 4: Gauge Charts Dinâmicos (Conectados aos resultados reais)
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os

def gerar_gauges_dinamicos():
    conn = get_connection()

    # Busca a polaridade média real de cada eixo
    query = """
        SELECT t.id, t.antidoto_referencia, AVG(vs.sentimento_num) as polaridade
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id != 3
        GROUP BY t.id
    """
    df_polaridade = pd.read_sql_query(query, conn)
    conn.close()

    # Cores temáticas para combinar com as Nuvens e o Stack Chart
    cores_eixos = {0: '#2E8B57', 1: '#D2691E', 2: '#4682B4'}

    for _, row in df_polaridade.iterrows():
        # Valor a ser exibido (usamos o valor real da polaridade)
        valor = row['polaridade']
        t_id = row['id']
        nome_limpo = row['antidoto_referencia'].split('(')[0].strip().lower()

        # --- Lógica do Gráfico ---
        limite_escala = 0.5
        # Garantimos que o arco mostre algo mesmo se for levemente negativo ou muito baixo
        valor_arco = max(0, valor)

        fatia_valor = (valor_arco / limite_escala) * 180
        fatia_restante = 180 - fatia_valor

        fig, ax = plt.subplots(figsize=(6, 3))

        # Aplicamos a cor temática do eixo
        cor_ativa = cores_eixos.get(t_id, '#50C878')
        ax.pie([fatia_valor, fatia_restante, 180],
               colors=[cor_ativa, '#F2F2F2', 'white'],
               startangle=180,
               counterclock=True,
               wedgeprops={'width': 0.4, 'edgecolor': 'white'})

        # Texto centralizado
        plt.text(0, 0.05, f"{valor:.3f}", ha='center', va='center',
                 fontsize=28, fontweight='bold', color='#333333')

        plt.text(0, -0.2, row['antidoto_referencia'], ha='center',
                 fontsize=10, color='#666666', fontweight='bold')

        ax.axis('equal')

        file_name = f"gauge_{nome_limpo}.jpg"
        plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✅ Gauge Dinâmico salvo: {file_name} (Valor: {valor:.3f})")

gerar_gauges_dinamicos()

In [ ]:
# Célula: Amplitude Emocional por Gênero (Mín, Máx e Média)
import pandas as pd
import matplotlib.pyplot as plt
import os

def gerar_grafico_amplitude_emocional():
    # Dados reais da sua consulta
    data = {
        'Genero': ['Apocalíptico', 'Epístola', 'Evangelho', 'Histórico', 'Pentateuco', 'Poético/Sapiencial', 'Profético'],
        'Min': [-0.978, -0.985, -0.989, -0.980, -0.990, -0.989, -0.991],
        'Max': [0.955, 0.979, 0.960, 0.957, 0.926, 0.991, 0.957],
        'Med': [0.144, 0.172, -0.163, 0.048, -0.209, 0.169, -0.393]
    }
    df = pd.DataFrame(data).sort_values(by='Med')

    plt.figure(figsize=(12, 7))

    # 1. Desenha a amplitude (do Min ao Max) como uma linha cinza sutil
    for i, row in df.iterrows():
        plt.plot([row['Min'], row['Max']], [row['Genero'], row['Genero']],
                 color='#D1D1D1', linewidth=4, alpha=0.6, zorder=1)

    # 2. Desenha a Média como um ponto colorido (Divergente)
    # Vermelho para médias negativas, Azul para médias positivas
    cores = ['#D9534F' if x < 0 else '#4A90E2' for x in df['Med']]
    plt.scatter(df['Med'], df['Genero'], color=cores, s=120, zorder=2, edgecolors='white')

    # 3. Linha vertical no Zero (Equilíbrio)
    plt.axvline(0, color='#333333', linestyle='--', linewidth=0.8, alpha=0.5)

    # Ajustes Minimalistas
    plt.xlabel('Amplitude de Sentimento (Mínimo, Máximo e Média)', fontsize=11)
    plt.ylabel('')
    plt.xlim(-1.1, 1.1) # Escala completa do BERTimbau

    sns.despine(left=True, bottom=False)
    plt.grid(False)
    plt.tight_layout()

    # Salvamento
    file_name = "amplitude_emocional_genero.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, format='jpg', bbox_inches='tight')

    print(f"✅ Gráfico de amplitude salvo: {file_name}")
    plt.show()

gerar_grafico_amplitude_emocional()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

def gerar_workflow_data_prep_destaque():
    # Proporção ampla para garantir que as fontes maximizadas respirem
    fig, ax = plt.subplots(figsize=(24, 7), facecolor='white')
    ax.set_facecolor('white')

    etapas = [
        {"n": "1", "titulo": "Ingestão", "sub": "Business Und.", "libs": "google.colab\nos, gc", "tipo": "und"},
        {"n": "2", "titulo": "Ambiente", "sub": "Data Prep", "libs": "transformers\ntorch", "tipo": "prep"},
        {"n": "3", "titulo": "Carga", "sub": "Data Prep", "libs": "sqlite3\npandas", "tipo": "prep"},
        {"n": "4", "titulo": "Limpeza", "sub": "Data Prep", "libs": "re\nstring", "tipo": "prep"},
        {"n": "5", "titulo": "Tópicos", "sub": "Modeling", "libs": "bertopic\nsklearn, nltk", "tipo": "mod"},
        {"n": "6", "titulo": "Sentimento", "sub": "Modeling", "libs": "pysentimiento\ntqdm", "tipo": "mod"},
        {"n": "7", "titulo": "Avaliação", "sub": "Evaluation", "libs": "matplotlib\nseaborn, wordcloud", "tipo": "eval"}
    ]

    # Esquema de cores refinado para diferenciar Data Prep
    cores = {
        "und":  {"face": "#F5F5F5", "edge": "#9E9E9E", "text": "#424242", "lib_color": "#616161"}, # Cinza
        "prep": {"face": "#FFF3E0", "edge": "#FF9800", "text": "#E65100", "lib_color": "#EF6C00"}, # Laranja (Data Prep)
        "mod":  {"face": "#E3F2FD", "edge": "#1976D2", "text": "#0D47A1", "lib_color": "#1565C0"}, # Azul
        "eval": {"face": "#E8F5E9", "edge": "#388E3C", "text": "#1B5E20", "lib_color": "#2E7D32"}  # Verde
    }

    n_etapas = len(etapas)
    box_w, box_h = 1.25, 0.95
    espacamento = 1.65

    # Linha conectora de fundo
    ax.plot([0, (n_etapas-1) * espacamento], [0.5, 0.5], color='#F0F0F0',
            linewidth=15, zorder=1, solid_capstyle='round')

    for i, etapa in enumerate(etapas):
        x = i * espacamento
        y = 0.5
        estilo = cores[etapa["tipo"]]

        # 1. Box da Etapa
        rect = patches.FancyBboxPatch(
            (x - box_w/2, y - box_h/2), box_w, box_h,
            boxstyle="round,pad=0.04", linewidth=2.8,
            edgecolor=estilo["edge"], facecolor=estilo["face"], zorder=3
        )
        ax.add_patch(rect)

        # 2. Rótulo Etapa X
        ax.text(x - box_w/2, y + box_h/2 + 0.08, f"Etapa {etapa['n']}",
                fontsize=13, fontweight='bold', color='#757575', ha='left')

        # 3. Título Principal (Max)
        ax.text(x, y + 0.25, etapa["titulo"], ha='center', va='center',
                fontsize=18, fontweight='black', color=estilo["text"], zorder=4)

        # 4. Subtítulo CRISP-DM
        ax.text(x, y + 0.08, etapa["sub"], ha='center', va='center',
                fontsize=12, style='italic', color=estilo["text"], alpha=0.9, zorder=4)

        # 5. Bibliotecas Monospace
        ax.text(x, y - 0.22, etapa["libs"], ha='center', va='center',
                fontsize=12, fontweight='bold', color=estilo["lib_color"],
                family='monospace', zorder=4)

        # 6. Setas (Cores seguem a origem do fluxo)
        if i < n_etapas - 1:
            ax.annotate("", xy=(x + espacamento - box_w/2 - 0.06, y),
                        xytext=(x + box_w/2 + 0.06, y),
                        arrowprops=dict(arrowstyle='-|>', color=estilo["edge"],
                        lw=2.5, mutation_scale=25), zorder=2)

    ax.set_xlim(-1.0, (n_etapas - 1) * espacamento + 1.0)
    ax.set_ylim(-0.1, 1.1)
    ax.axis('off')

    plt.tight_layout()

    file_name = "workflow_crisp_data_prep_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight', pad_inches=0.1)

    print(f"✅ Workflow com Data Prep destacado gerado: {file_name}")
    plt.show()

gerar_workflow_data_prep_destaque()